In [1]:
import sys
import json
import torch
import numpy as np

from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent))
from config import data_config
from config import preprocess_config

from preprocessing.preprocess_prices import preprocess_prices
from preprocessing.haar_wavelet import HaarWaveletTransform

In [2]:
"""
Clipped -> Preprocessed
"""

Path("../data/preprocessed").mkdir(parents=True, exist_ok=True)

# IR dictionary 
with open(f'../data/raw/daily_ir.json', 'r') as f:
    ir = json.load(f)
ir_dict = {entry['date']: float(entry['value']) for entry in ir}

# VIX dictionary
with open(f'../data/raw/daily_vix.json', 'r') as f:
    vix = json.load(f)
vix_dict = {entry['Date']: entry['Close'] for entry in vix}

# Preprocess data
count = 0
for ticker in data_config.TICKERS:
    # Load JSON file
    with open(f'../data/raw/daily_data_{ticker}.json', 'r') as f:
        data = json.load(f)
    
    # Extract time series
    time_series = data['Time Series (Daily)']
    dates = sorted(time_series.keys())
    dates = [date for date in dates if data_config.WINDOW_START <= date <= data_config.WINDOW_END] # Window
    closing_prices = torch.tensor([float(time_series[date]['5. adjusted close']) for date in dates]) # Change based on function
    
    # Loop through T - len(window) + 1
    ticker_samples = []
    for start in range(0, len(closing_prices) - preprocess_config.T, preprocess_config.STRIDE):

        # Returns, trend, realized vol for this window
        returns, trend, realized_vol = preprocess_prices(closing_prices, start=start)
        
        # Dates for this window
        window_dates = dates[start:start + preprocess_config.T]
        
        # Average interest rate for this window
        avg_ir = np.mean([ir_dict[date] for date in window_dates if date in ir_dict])
        
        # Average volatility index for this window
        avg_vix = np.mean([vix_dict[date] for date in window_dates if date in vix_dict])
        
        # Sample dictionary
        sample = {
            "returns": returns.tolist(),
            "trend": trend.item(),
            "realized_vol": realized_vol.item(),
            "interest_rate": avg_ir,
            "volatility_index": avg_vix
        }
        ticker_samples.append(sample)

        count += 1
    
    # Save to JSON file
    output_path = f'../data/preprocessed/prep_data_{ticker}.json'
    with open(output_path, 'w') as f:
        json.dump(ticker_samples, f, indent=2)
    
    print(f"{ticker}: {len(ticker_samples)} samples saved to {output_path}")

    
print(f"\nTotal samples across all tickers: {count}")


NVDA: 78 samples saved to ../data/preprocessed/prep_data_NVDA.json
AAPL: 78 samples saved to ../data/preprocessed/prep_data_AAPL.json
GOOG: 22 samples saved to ../data/preprocessed/prep_data_GOOG.json
MSFT: 78 samples saved to ../data/preprocessed/prep_data_MSFT.json
AMZN: 78 samples saved to ../data/preprocessed/prep_data_AMZN.json
META: 29 samples saved to ../data/preprocessed/prep_data_META.json
AVGO: 40 samples saved to ../data/preprocessed/prep_data_AVGO.json
TSLA: 37 samples saved to ../data/preprocessed/prep_data_TSLA.json
BRK-B: 78 samples saved to ../data/preprocessed/prep_data_BRK-B.json
WMT: 78 samples saved to ../data/preprocessed/prep_data_WMT.json
LLY: 78 samples saved to ../data/preprocessed/prep_data_LLY.json
JPM: 78 samples saved to ../data/preprocessed/prep_data_JPM.json
XOM: 78 samples saved to ../data/preprocessed/prep_data_XOM.json
JNJ: 78 samples saved to ../data/preprocessed/prep_data_JNJ.json
V: 46 samples saved to ../data/preprocessed/prep_data_V.json
MU: 78 sa

In [3]:
"""
Preprocessed -> Upsampled
"""

Path("../data/upsampled").mkdir(parents=True, exist_ok=True)

all_samples = []
for ticker in data_config.TICKERS:
    # Load JSON file
    with open(f'../data/preprocessed/prep_data_{ticker}.json', 'r') as f:
        all_samples.extend(json.load(f))

print(f"Total samples before upsampling: {len(all_samples)}")

# Absolute trends
abs_trends = [abs(sample['trend']) for sample in all_samples]

# Percentiles (75th, 90th, 95th)
threshold_25 = np.percentile(abs_trends, 75)
threshold_10 = np.percentile(abs_trends, 90)
threshold_5 = np.percentile(abs_trends, 95)

# Apply upsampling 
upsampled_data = []

for i in range(len(all_samples)):
    sample = all_samples[i]
    abs_trend = abs_trends[i]
    
    # Replicate according to tier
    if abs_trend >= threshold_5:
        for _ in range(5):
            upsampled_data.append(sample)
    elif abs_trend >= threshold_10:
        for _ in range(5):
            upsampled_data.append(sample)
    elif abs_trend >= threshold_25:
        for _ in range(5):
            upsampled_data.append(sample)
    else:
        upsampled_data.append(sample)

# Save to single JSON file
output_path = '../data/upsampled/upsampled_data.json'
with open(output_path, 'w') as f:
    json.dump(upsampled_data, f, indent=2)

print(f"Total samples after upsampling: {len(upsampled_data)}")

Total samples before upsampling: 6804
Total samples after upsampling: 13608


In [4]:
"""
Upsampled -> Train (2D)
"""

Path("../data/train").mkdir(parents=True, exist_ok=True)

# Load JSON file
with open('../data/upsampled/upsampled_data.json', 'r') as f:
    data = json.load(f)

print(f"Loaded {len(data)} samples...")

transformed_data = []
for sample in data:
    # Convert returns list to torch tensor
    returns_1d = torch.tensor(sample['returns'], dtype=torch.float32)
    
    # 1D to 2D Haar Wavelet Transform
    haar_transform = HaarWaveletTransform()
    returns_2d = haar_transform.forward(returns_1d)
    
    # Nested list for easier torch.tensor conversion
    returns_2d_list = returns_2d.squeeze(0).tolist()
    
    # Create new sample with 2D returns
    transformed_sample = {
        "returns_2d": returns_2d_list,
        "trend": sample['trend'],
        "realized_vol": sample['realized_vol'],
        "interest_rate": sample['interest_rate'],
        "volatility_index": sample['volatility_index']
    }
    transformed_data.append(transformed_sample)

print(f"Transformed {len(transformed_data)} samples with shape {len(transformed_data[0]['returns_2d'])}x{len(transformed_data[0]['returns_2d'][0])}")

# Save transformed data
output_path = '../data/train/train_data_2d.json'
with open(output_path, 'w') as f:
    json.dump(transformed_data, f, indent=2)

print(f"Saved transformed data to {output_path}")
    

Loaded 13608 samples...
Transformed 13608 samples with shape 8x8
Saved transformed data to ../data/train/train_data_2d.json
